In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [2]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})
        page = tif.pages[0]

        def _rational(name):
            tag = page.tags.get(name)
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = _rational("XResolution")
        yres = _rational("YResolution")
        resunit_tag = page.tags.get("ResolutionUnit")
        resunit = int(resunit_tag.value) if resunit_tag is not None else None

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta


def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata.

    The channel / slice / frame counts are recomputed from `arr` (rather than
    copied from the source metadata) so the ImageJ header stays correct even
    when the channel count changes, e.g. when we add a 4th `valid` channel to a
    previously 3-channel stack.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    # Carry through acquisition metadata that does not depend on array shape.
    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    # Recompute shape-dependent counts from the array actually being written.
    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    tifffile.imwrite(path, arr, **kwargs)


In [3]:
# Curate the segmentation channel of each tif and record a VALIDITY ("valid")
# channel alongside it. A single napari viewer is open at any moment. Edit the
# mask, then click "Save & Next" (or close the window); the viewer reopens with
# the next image automatically.
#
# Every output is a 4-channel tif: [brightfield, fluorescence, mask, valid].
# The `valid` channel marks the voxels a human vouches for (valid == 1 -> use in
# the training loss; valid == 0 -> ignore), so the stack can be curated sparsely
# without poisoning training with uncurated slices.
#
# Two curation modes are supported:
#   - mode="middle": only the middle z-slice is shown/edited in 2D. The curated
#     middle-slice mask is duplicated across every z-slice to form a prism, and
#     `valid` is set to 1 on the middle slice only (the slice actually curated)
#     and 0 elsewhere. Saved as a NEW 4-channel tif in `output_folder`.
#   - mode="stack": the full 3D stack is shown/edited (napari z-slider). You fix
#     the mask in 3D and mark which voxels are trustworthy in the `valid` layer.
#     You never have to draw twice:
#       * any paint/erase stroke on the MASK layer auto-marks that footprint as
#         valid on the current slice;
#       * the "Mark current slice valid" button flips the WHOLE current z-slice
#         to valid (vessel + background) in one click.
#     Saved as a NEW 4-channel tif in `output_folder`.
#
# Pixel-size / spacing metadata is preserved either way.
#
# In a notebook the Qt event loop is already running, so we can't use a
# blocking `for` loop (that opens every viewer at once). Instead we chain
# images together through the viewer's close event.

class MaskCurator:
    def __init__(self, images, mode="middle", start_index=0, output_folder=None,
                 brightfield_channel=0, fluorescence_channel=1, mask_channel=2,
                 valid_channel=3):
        if mode not in ("middle", "stack"):
            raise ValueError("mode must be 'middle' or 'stack'")
        if output_folder is None:
            raise ValueError("output_folder is required")
        self.images = images
        self.mode = mode
        self.index = start_index
        self.output_folder = Path(output_folder)
        self.output_folder.mkdir(parents=True, exist_ok=True)
        self.brightfield_channel = brightfield_channel
        self.fluorescence_channel = fluorescence_channel
        self.mask_channel = mask_channel
        self.valid_channel = valid_channel

        self.viewer = None
        self.arr = None
        self.meta = None
        self.mask_layer = None
        self.valid_layer = None
        self.mid_z = None
        self._mask_snapshot = None
        self._orig_close = None
        self._advancing = False

    def start(self):
        self._open_current()

    def _open_current(self):
        if self.index >= len(self.images):
            print("All images curated.")
            return

        path = self.images[self.index]
        print(f"[{self.index + 1}/{len(self.images)}] Curating {path.name}")

        self.arr, self.meta = read_image_and_meta(path)
        n_z = self.arr.shape[0]
        n_c = self.arr.shape[1]
        self.mid_z = n_z // 2

        if self.mode == "middle":
            self._open_middle(path)
        else:
            self._open_stack(path, n_c)

        # Route the window close (button or X) through our save+advance logic.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    def _open_middle(self, path):
        # Middle slice of each channel (2D Y, X).
        brightfield = self.arr[self.mid_z, self.brightfield_channel, :, :]
        fluorescence = self.arr[self.mid_z, self.fluorescence_channel, :, :]
        mask = self.arr[self.mid_z, self.mask_channel, :, :]
        title = (f"[{self.index + 1}/{len(self.images)}] {path.name} "
                 f"(z={self.mid_z})")

        self.viewer = napari.Viewer(title=title)
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive")
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive")
        self.mask_layer = self.viewer.add_labels(mask.astype(np.int32),
                                                 name="mask")
        self.valid_layer = None

        next_btn = PushButton(text="Save & Next")
        next_btn.clicked.connect(self.viewer.close)
        self.viewer.window.add_dock_widget(next_btn, area="right",
                                           name="curation")

    def _open_stack(self, path, n_c):
        # Full 3D stack of each channel (3D Z, Y, X).
        brightfield = self.arr[:, self.brightfield_channel, :, :]
        fluorescence = self.arr[:, self.fluorescence_channel, :, :]
        mask = self.arr[:, self.mask_channel, :, :]
        if n_c > self.valid_channel:
            valid = self.arr[:, self.valid_channel, :, :]
        else:
            valid = np.zeros_like(mask)
        title = (f"[{self.index + 1}/{len(self.images)}] {path.name} "
                 f"(3D stack)")

        self.viewer = napari.Viewer(title=title)
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive")
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive")
        self.mask_layer = self.viewer.add_labels(mask.astype(np.int32),
                                                 name="mask")
        self.valid_layer = self.viewer.add_labels(valid.astype(np.int32),
                                                  name="valid", opacity=0.4)

        # Paint on the mask by default; the valid layer fills itself (below).
        self.viewer.layers.selection.active = self.mask_layer

        # Auto-mirror: any edit to the mask marks that footprint valid on the
        # current slice, so you never draw the same region twice. We diff the
        # current slice against a snapshot (version-proof vs. napari paint APIs).
        self._mask_snapshot = self.mask_layer.data.copy()
        paint_event = getattr(self.mask_layer.events, "paint", None)
        if paint_event is None:
            paint_event = getattr(self.mask_layer.events, "set_data", None)
        if paint_event is not None:
            paint_event.connect(self._on_mask_paint)

        mark_btn = PushButton(text="Mark current slice valid")
        mark_btn.clicked.connect(self._mark_slice_valid)
        self.viewer.window.add_dock_widget(mark_btn, area="right",
                                           name="validity")

        next_btn = PushButton(text="Save & Next")
        next_btn.clicked.connect(self.viewer.close)
        self.viewer.window.add_dock_widget(next_btn, area="right",
                                           name="curation")

    def _current_z(self):
        return int(self.viewer.dims.current_step[0])

    def _on_mask_paint(self, event=None):
        # Mirror just-edited mask voxels (paint OR erase) into `valid` on the
        # slice currently displayed, then refresh the snapshot for that slice.
        if self.valid_layer is None:
            return
        z = self._current_z()
        changed = self.mask_layer.data[z] != self._mask_snapshot[z]
        if changed.any():
            self.valid_layer.data[z][changed] = 1
            self._mask_snapshot[z] = self.mask_layer.data[z].copy()
            self.valid_layer.refresh()

    def _mark_slice_valid(self):
        # Vouch for the entire current slice (vessel + background) in one click.
        if self.valid_layer is None:
            return
        z = self._current_z()
        self.valid_layer.data[z, :, :] = 1
        self.valid_layer.refresh()

    def _assemble_output(self, mask_full, valid_full):
        """Build a 4-channel (Z, 4, Y, X) array = [bf, fluo, mask, valid]."""
        n_z, _, ny, nx = self.arr.shape
        out = np.zeros((n_z, 4, ny, nx), dtype=self.arr.dtype)
        out[:, 0] = self.arr[:, self.brightfield_channel]
        out[:, 1] = self.arr[:, self.fluorescence_channel]
        out[:, 2] = mask_full
        out[:, 3] = valid_full
        return out

    def _on_close(self, event):
        if not self._advancing:
            self._advancing = True
            path = self.images[self.index]
            n_z = self.arr.shape[0]

            if self.mode == "middle":
                # Curated 2D middle slice -> prism across every z.
                curated = self.mask_layer.data.astype(self.arr.dtype)
                mask_full = np.broadcast_to(curated, (n_z,) + curated.shape)
                # Only the curated middle slice is vouched for.
                valid_full = np.zeros((n_z,) + curated.shape,
                                      dtype=self.arr.dtype)
                valid_full[self.mid_z] = 1
            else:
                # Curated full 3D mask + validity layer.
                mask_full = self.mask_layer.data.astype(self.arr.dtype)
                valid_full = (self.valid_layer.data > 0).astype(self.arr.dtype)

            out = self._assemble_output(mask_full, valid_full)
            self.meta["axes"] = "ZCYX"

            out_path = self.output_folder / path.name
            save_image(out_path, out, self.meta)
            print(f"    saved -> {out_path}")

            self.index += 1
            # Open the next image once this window has finished closing.
            QTimer.singleShot(200, self._open_next)
        self._orig_close(event)

    def _open_next(self):
        self._advancing = False
        self._open_current()


In [ ]:
vascumap_masks_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\masks_to_curate")
expanded_middle_slice_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks")
curated_3d_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_3d")
expanded_middle_slice_folder.mkdir(parents=True, exist_ok=True)
curated_3d_folder.mkdir(parents=True, exist_ok=True)

images = sorted(vascumap_masks_folder.glob("*.tif"))
images = images[10:]
print(f"{len(images)} images found")


58 images found


## Option 1 — Curate the middle slice, save to output folder

Opens the **middle z-slice** of each image in 2D. The curated middle-slice
mask is expanded across the full z-stack (prism) and saved as a **new 4-channel
tif** `[brightfield, fluorescence, mask, valid]` in `expanded_middle_slice_folder`.
The `valid` channel is set to 1 on the **middle slice only** (the slice actually
curated) and 0 elsewhere. Brightfield and fluorescence channels are left
unchanged.


In [ ]:
# Skip images that have already been curated (already in the output folder).
already_curated = {p.name for p in expanded_middle_slice_folder.glob("*.tif")}
images_to_curate = [p for p in images if p.name not in already_curated]
print(f"{len(images_to_curate)} images to curate "
      f"({len(images) - len(images_to_curate)} already curated, skipped)")

# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(
    images_to_curate,
    mode="middle",
    output_folder=expanded_middle_slice_folder,
    start_index=0,
)
curator.start()


## Option 2 — Curate the full 3D stack, save to the 3D output folder

Opens the **full 3D stack** of each image (use the napari z-slider to move
through slices). Fix the **mask** layer normally; the **valid** layer records
which voxels you vouch for and fills itself so you never draw twice:

- any **paint/erase stroke on the mask** auto-marks that footprint valid on the
  current slice;
- the **"Mark current slice valid"** button flips the whole current z-slice to
  valid (vessel *and* background) in one click.

Reads the 4-channel prism tifs from `expanded_middle_slice_folder` and saves the
curated 4-channel result `[brightfield, fluorescence, mask, valid]` to
`curated_3d_folder`. Brightfield and fluorescence channels are left unchanged.


In [ ]:
# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(
    images=sorted(expanded_middle_slice_folder.glob("*.tif")),
    mode="stack",
    output_folder=curated_3d_folder,
    start_index=0,
)
curator.start()


[1/9] Curating 20260514_FL37_ARi_device1_infocus.tif


    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device1_infocus.tif
[2/9] Curating 20260514_FL37_ARi_device4_infocus.tif
    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device4_infocus.tif
[3/9] Curating 20260514_FL37_UTD_device2_infocus.tif
    saved -> C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_UTD_device2_infocus.tif
[4/9] Curating 20260514_FL37_UTD_device4_infocus.tif
